# Integrated Photonics — First Terms
**Waveguide modes · coupling · dispersion · ring resonators · coherent detection**

Runs in **Google Colab** or locally.  Covers the foundational vocabulary you need before diving into photonic integrated circuits (PICs), GS phase retrieval, and coherent receivers.

| § | Topic |
|---|-------|
| 1 | Slab waveguide modes (TE/TM) |
| 2 | Effective index + confinement |
| 3 | Waveguide dispersion |
| 4 | Directional coupler (splitting ratio) |
| 5 | 90° hybrid — coherent detection |
| 6 | Ring resonator (FSR, Q, finesse) |
| 7 | Group velocity dispersion in Si waveguides |
| 8 | Mach-Zehnder modulator (EO effect) |

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('pip install -q numpy matplotlib scipy sympy')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import brentq
from scipy.constants import c, h, hbar, e
from sympy import *

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print('Ready. λ_telecom = 1310 nm (O-band), 1550 nm (C-band)')

---
## §1 · Slab waveguide modes (TE)

For a symmetric slab with core index $n_1$, cladding $n_2$, thickness $d$, the TE eigenvalue equation is:

$$\kappa d = m\pi + 2\arctan\!\left(\frac{\gamma}{\kappa}\right), \quad \kappa = \sqrt{n_1^2 k_0^2 - \beta^2}, \quad \gamma = \sqrt{\beta^2 - n_2^2 k_0^2}$$

Guided modes exist for $n_2 k_0 < \beta < n_1 k_0$.

In [ ]:
def slab_modes(n1, n2, d_um, lam_um=1.55, max_modes=6):
    """
    Find TE mode effective indices for symmetric slab waveguide.
    Returns list of (m, n_eff, kappa, gamma) tuples.
    """
    k0 = 2 * np.pi / lam_um
    d  = d_um
    modes = []
    for m in range(max_modes):
        # TE eigenvalue: kappa*d/2 - m*pi/2 = arctan(gamma/kappa)
        def eig_eq(beta):
            if beta <= n2 * k0 or beta >= n1 * k0:
                return 1e10
            kappa = np.sqrt(n1**2 * k0**2 - beta**2)
            gamma = np.sqrt(beta**2 - n2**2 * k0**2)
            return kappa * d / 2 - m * np.pi / 2 - np.arctan(gamma / kappa)

        b_lo = n2 * k0 + 1e-6
        b_hi = n1 * k0 - 1e-6
        try:
            beta_m = brentq(eig_eq, b_lo, b_hi)
            kappa  = np.sqrt(n1**2 * k0**2 - beta_m**2)
            gamma  = np.sqrt(beta_m**2 - n2**2 * k0**2)
            modes.append((m, beta_m / k0, kappa, gamma))
        except ValueError:
            break  # no more guided modes
    return modes

# Silicon nitride waveguide: n_core=2.0 (Si3N4), n_clad=1.45 (SiO2)
n1, n2 = 2.0, 1.45
d_vals = np.linspace(0.1, 3.0, 200)

# Count modes vs waveguide thickness
n_modes = [len(slab_modes(n1, n2, d)) for d in d_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].step(d_vals, n_modes, where='post', lw=2, color='#377eb8')
axes[0].axhline(1, ls='--', color='gray', alpha=0.5)
for d_cut in [lam / (2 * np.sqrt(n1**2 - n2**2)) for lam in [1.55]]:
    axes[0].axvline(d_cut, ls=':', color='red', alpha=0.7, label=f'V=π/2 cutoff d={d_cut:.2f}μm')
axes[0].set_xlabel('Core thickness d (μm)'); axes[0].set_ylabel('Number of TE modes')
axes[0].set_title(f'Si₃N₄/SiO₂ slab: mode count vs thickness  (λ=1.55 μm)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Mode field profiles for d=0.8 μm
d_wg = 0.8  # μm
modes = slab_modes(n1, n2, d_wg, max_modes=3)
x = np.linspace(-1.5, 1.5, 800)  # μm
k0 = 2 * np.pi / 1.55

for m_idx, (m, neff, kappa, gamma) in enumerate(modes):
    psi = np.zeros_like(x)
    A = 1.0
    for i, xi in enumerate(x):
        if abs(xi) <= d_wg / 2:
            psi[i] = A * np.cos(kappa * xi - m * np.pi / 2)
        else:
            sign = np.sign(xi)
            psi[i] = A * np.cos(kappa * d_wg/2 - m*np.pi/2) * np.exp(-gamma * (abs(xi) - d_wg/2))
    psi /= np.max(np.abs(psi))
    axes[1].plot(x, psi + m_idx * 2.5, label=f'TE{m}  neff={neff:.4f}')

axes[1].axvspan(-d_wg/2, d_wg/2, alpha=0.1, color='blue', label='Core')
axes[1].set_xlabel('Position x (μm)'); axes[1].set_ylabel('Normalized field (offset)')
axes[1].set_title(f'TE mode profiles  d={d_wg} μm, Si₃N₄/SiO₂')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('ip_slab_modes.png', bbox_inches='tight'); plt.show()
for m, neff, kappa, gamma in modes:
    print(f'TE{m}: n_eff={neff:.5f}  κ={kappa:.3f}/μm  γ={gamma:.3f}/μm')

## §2 · Confinement factor + single-mode condition

The **V-number** determines how many modes are guided:
$$V = \frac{\pi d}{\lambda}\sqrt{n_1^2 - n_2^2}$$

Single-mode condition: $V < \pi/2$ (slab) or $V < 2.405$ (circular fiber).

**Confinement factor** $\Gamma$ = fraction of power in the core — key for gain in active waveguides.

In [ ]:
NA_vals   = [np.sqrt(n1**2 - n2**2) for n1, n2 in [(2.0, 1.45), (3.48, 1.45), (1.5, 1.44)]]
labels    = ['Si₃N₄/SiO₂', 'Si/SiO₂', 'SiO₂/SiO₂ (Δn=0.3%)']
colors_V  = ['#377eb8', '#e41a1c', '#4daf4a']

d_range = np.linspace(0.05, 2.0, 400)
lam_wg  = 1.55

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for NA, lab, col in zip(NA_vals, labels, colors_V):
    V = np.pi * d_range / lam_wg * NA
    axes[0].plot(d_range, V, color=col, lw=2, label=lab)

axes[0].axhline(np.pi/2, ls='--', color='gray', label='V=π/2 (slab SM cutoff)')
axes[0].axhline(2.405,   ls=':',  color='black', label='V=2.405 (fiber SM cutoff)')
axes[0].set_xlabel('Core width d (μm)'); axes[0].set_ylabel('V number')
axes[0].set_title('V number vs waveguide width (λ=1.55 μm)')
axes[0].legend(fontsize=8); axes[0].set_ylim(0, 10); axes[0].grid(True, alpha=0.3)

# Confinement factor vs V (approximate analytic expression)
V_range = np.linspace(0.01, 5, 400)
# Marcuse approximation for fundamental mode
Gamma_approx = 1 - 2 / (V_range**2 + 1)  # heuristic; exact requires mode integrals
axes[1].plot(V_range, np.clip(Gamma_approx, 0, 1), lw=2, color='crimson')
axes[1].axvline(np.pi/2, ls='--', color='gray', label='Slab SM cutoff')
axes[1].axhline(0.8, ls=':', color='blue', alpha=0.5, label='Γ=0.8 (typical target)')
axes[1].set_xlabel('V number'); axes[1].set_ylabel('Confinement factor Γ')
axes[1].set_title('Confinement factor (TE₀) — Marcuse approximation')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('ip_confinement.png', bbox_inches='tight'); plt.show()

## §3 · Waveguide dispersion

Total dispersion = material dispersion + waveguide dispersion:
$$D_{\text{total}} = D_{\text{mat}} + D_{\text{wg}} \quad [\text{ps/(nm·km)}]$$

**Key:** Silicon waveguides have anomalous dispersion above ~1300 nm — enabling soliton propagation and supercontinuum generation.  For GS phase retrieval, large $|D|$ is required.

In [ ]:
# Sellmeier coefficients for Si (wavelength in μm)
def n_si(lam_um):
    """Silicon refractive index via Sellmeier (Green, 1995)."""
    return np.sqrt(1 + 10.6684293 * lam_um**2 / (lam_um**2 - 0.3015516**2)
                     + 0.0030434748 * lam_um**2 / (lam_um**2 - 1.13475115**2)
                     + 1.54133408 * lam_um**2 / (lam_um**2 - 1104**2))

def n_sio2(lam_um):
    """SiO₂ Sellmeier (Malitson, 1965)."""
    return np.sqrt(1 + 0.6961663 * lam_um**2 / (lam_um**2 - 0.0684043**2)
                     + 0.4079426 * lam_um**2 / (lam_um**2 - 0.1162414**2)
                     + 0.8974994 * lam_um**2 / (lam_um**2 - 9.896161**2))

lam_um = np.linspace(1.0, 2.0, 500)
lam_nm = lam_um * 1e3

n_si_vals   = n_si(lam_um)
n_sio2_vals = n_sio2(lam_um)

# Material dispersion D = -(lambda/c) d²n/dlambda²  [ps/(nm·km)]
dlam = lam_um[1] - lam_um[0]
def material_dispersion(n_vals, lam_um_arr):
    d2n = np.gradient(np.gradient(n_vals, lam_um_arr), lam_um_arr)
    return -(lam_um_arr / (c * 1e-12)) * d2n * 1e9 / 1e3  # ps/(nm·km)

D_mat_si   = material_dispersion(n_si_vals, lam_um)
D_mat_sio2 = material_dispersion(n_sio2_vals, lam_um)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(lam_nm, n_si_vals,   lw=2, color='#e41a1c', label='Si')
axes[0].plot(lam_nm, n_sio2_vals, lw=2, color='#377eb8', label='SiO₂')
axes[0].set_xlabel('Wavelength (nm)'); axes[0].set_ylabel('Refractive index n')
axes[0].set_title('Sellmeier indices — Si and SiO₂'); axes[0].legend()
axes[0].axvline(1310, ls='--', color='gray', alpha=0.6, label='1310 nm')
axes[0].axvline(1550, ls=':',  color='gray', alpha=0.6, label='1550 nm')
axes[0].grid(True, alpha=0.3)

axes[1].plot(lam_nm, D_mat_si,   lw=2, color='#e41a1c', label='Si (material)')
axes[1].plot(lam_nm, D_mat_sio2, lw=2, color='#377eb8', label='SiO₂ (material)')
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].axvline(1310, ls='--', color='gray', alpha=0.6)
axes[1].axvline(1550, ls=':',  color='gray', alpha=0.6)
axes[1].fill_between(lam_nm, D_mat_si, 0,
    where=(D_mat_si > 0), alpha=0.15, color='red', label='Anomalous')
axes[1].fill_between(lam_nm, D_mat_si, 0,
    where=(D_mat_si < 0), alpha=0.15, color='blue', label='Normal')
axes[1].set_xlabel('Wavelength (nm)'); axes[1].set_ylabel('D [ps/(nm·km)]')
axes[1].set_title('Material dispersion — GVD zero-crossing')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-3000, 3000)

plt.tight_layout(); plt.savefig('ip_dispersion.png', bbox_inches='tight'); plt.show()

# ZDW for Si
try:
    zdw_idx = np.where(np.diff(np.sign(D_mat_si)))[0][0]
    print(f'Si material dispersion ZDW ≈ {lam_nm[zdw_idx]:.0f} nm')
except IndexError:
    print('ZDW not found in this wavelength range')

## §4 · Directional coupler — power splitting

Two parallel waveguides exchange power via evanescent coupling:
$$P_1(z) = \cos^2(\kappa z), \quad P_2(z) = \sin^2(\kappa z)$$

$\kappa$ = coupling coefficient (depends on gap, overlap integral).  A 3 dB (50/50) splitter uses $\kappa L = \pi/4$.

In [ ]:
kappa = 1.0  # normalized coupling coefficient
z = np.linspace(0, 3 * np.pi / (2 * kappa), 500)

P1 = np.cos(kappa * z)**2
P2 = np.sin(kappa * z)**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(z / (np.pi / kappa), P1, lw=2, color='#377eb8', label='Port 1 (through)')
axes[0].plot(z / (np.pi / kappa), P2, lw=2, color='#e41a1c', label='Port 2 (cross)')
axes[0].axhline(0.5, ls='--', color='gray', label='3 dB (50/50)')
axes[0].axvline(0.25, ls=':', color='green', alpha=0.8, label='3dB coupler length')
axes[0].axvline(0.5,  ls=':', color='purple', alpha=0.8, label='Full cross-over')
axes[0].set_xlabel('Propagation z (units of L_π)'); axes[0].set_ylabel('Power fraction')
axes[0].set_title('Directional coupler power transfer'); axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Splitting ratio vs coupling length
gap_um = np.linspace(0.1, 1.0, 300)
# Coupling coefficient decays exponentially with gap (evanescent field)
kappa_gap = 2.0 * np.exp(-5 * gap_um)  # heuristic: kappa ~ exp(-gamma*gap)
L_3dB = np.pi / (4 * kappa_gap)  # length for 50/50 split

axes[1].semilogy(gap_um, L_3dB, lw=2, color='crimson')
axes[1].set_xlabel('Coupler gap (μm)'); axes[1].set_ylabel('3 dB coupling length L (μm)')
axes[1].set_title('3 dB coupler length vs gap (evanescent coupling)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(10, ls='--', color='gray', label='L=10 μm')
axes[1].legend()

plt.tight_layout(); plt.savefig('ip_coupler.png', bbox_inches='tight'); plt.show()

## §5 · 90° optical hybrid — coherent detection

The **90° hybrid** is the heart of coherent receivers and the GS phase retrieval system.  It mixes signal $E_s$ and local oscillator $E_{LO}$ with 0°/90° phase shifts:

$$\begin{pmatrix}E_1\\E_2\\E_3\\E_4\end{pmatrix} = \frac{1}{2}\begin{pmatrix}1&1\\1&-1\\1&j\\1&-j\end{pmatrix}\begin{pmatrix}E_s\\E_{LO}\end{pmatrix}$$

Balanced detection: $I \propto \text{Re}[E_s E_{LO}^*]$ (I channel) and $Q \propto \text{Im}[E_s E_{LO}^*]$ (Q channel).

In [ ]:
# Simulate coherent QPSK receiver with 90° hybrid
rng = np.random.default_rng(42)
N_sym = 500
SNR_dB = 20.0

# QPSK symbols: phase ∈ {π/4, 3π/4, 5π/4, 7π/4}
phases   = rng.choice([1, 3, 5, 7], N_sym) * np.pi / 4
Es_amp   = 1.0
E_s      = Es_amp * np.exp(1j * phases)  # signal field
E_LO     = 1.0 * np.ones(N_sym)          # ideal LO (zero phase)

# Add noise
sigma = Es_amp / np.sqrt(2 * 10**(SNR_dB / 10))
E_s  += sigma * (rng.normal(size=N_sym) + 1j * rng.normal(size=N_sym))

# 90° hybrid outputs
E1 = (E_s + E_LO) / 2
E2 = (E_s - E_LO) / 2
E3 = (E_s + 1j * E_LO) / 2
E4 = (E_s - 1j * E_LO) / 2

# Balanced detection
I_ch = np.abs(E1)**2 - np.abs(E2)**2  # ∝ Re[E_s * E_LO*]
Q_ch = np.abs(E3)**2 - np.abs(E4)**2  # ∝ Im[E_s * E_LO*]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(I_ch, Q_ch, s=8, alpha=0.5, c='#377eb8')
axes[0].set_aspect('equal')
axes[0].set_xlabel('I (in-phase)'); axes[0].set_ylabel('Q (quadrature)')
axes[0].set_title(f'QPSK constellation — 90° hybrid, SNR={SNR_dB} dB')
for phi in [1, 3, 5, 7]:
    ax = np.cos(phi * np.pi / 4); ay = np.sin(phi * np.pi / 4)
    axes[0].plot(ax, ay, 'r+', markersize=14, markeredgewidth=2)
axes[0].grid(True, alpha=0.3)

# Phase recovery: estimated phase vs true phase
phi_est  = np.angle(I_ch + 1j * Q_ch)
phi_true = phases
delta    = np.angle(np.exp(1j * (phi_est - phi_true)))
rms_deg  = np.degrees(np.sqrt(np.mean(delta**2)))

axes[1].hist(np.degrees(delta), bins=60, color='crimson', alpha=0.7, density=True)
axes[1].set_xlabel('Phase error (°)'); axes[1].set_ylabel('Probability density')
axes[1].set_title(f'Phase error distribution  RMS={rms_deg:.2f}°')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('ip_hybrid.png', bbox_inches='tight'); plt.show()
print(f'QPSK coherent receiver: RMS phase error = {rms_deg:.2f}°  (SNR={SNR_dB} dB)')

## §6 · Ring resonator — FSR, Q factor, finesse

Resonance condition: $n_{eff} \cdot 2\pi R = m \lambda$

$$\text{FSR} = \frac{\lambda^2}{n_g \cdot 2\pi R}, \quad Q = \frac{\lambda}{\Delta\lambda}, \quad \mathcal{F} = \frac{\text{FSR}}{\Delta\lambda}$$

**EE link:** ring resonators are used as WDM filters, modulators, and dispersion engineering elements in PICs.  Q factors of $10^6$–$10^8$ achieved in Si₃N₄.

In [ ]:
def ring_transmission(lam_nm, lam0_nm, R_um, n_eff, ng, a_round, t_coupler):
    """
    All-pass ring resonator transmission (Yariv 2000).
    a_round: round-trip field amplitude loss (a=1: lossless)
    t_coupler: self-coupling coefficient (t²+kappa²=1)
    """
    phi = 2 * np.pi * n_eff * 2 * np.pi * R_um * 1e-6 / (lam_nm * 1e-9)
    T = (a_round**2 - 2 * a_round * t_coupler * np.cos(phi) + t_coupler**2) / \
        (1 - 2 * a_round * t_coupler * np.cos(phi) + (a_round * t_coupler)**2)
    return T

R_um   = 50.0    # ring radius μm
n_eff  = 1.8     # Si3N4 TE0 effective index
ng     = 2.0     # group index
a_rt   = 0.99    # round-trip field amplitude (loss = 2%)
t_c    = 0.95    # self-coupling

lam0 = 1550.0   # center wavelength
FSR_nm = lam0**2 / (ng * 2 * np.pi * R_um * 1e3)  # nm

lam_scan = np.linspace(lam0 - 2*FSR_nm, lam0 + 2*FSR_nm, 5000)
T = ring_transmission(lam_scan, lam0, R_um, n_eff, ng, a_rt, t_c)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(lam_scan - lam0, 10*np.log10(T + 1e-12), color='#377eb8', lw=1.5)
axes[0].set_xlabel('Wavelength offset Δλ (nm)'); axes[0].set_ylabel('Transmission (dB)')
axes[0].set_title(f'Ring resonator  R={R_um}μm, a={a_rt}, t={t_c}')
axes[0].axhline(-3, ls='--', color='gray', label='-3 dB')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Q vs ring radius sweep
R_sweep = np.logspace(1, 3, 200)  # 10 μm to 1 mm
FSR_sweep = lam0**2 / (ng * 2 * np.pi * R_sweep * 1e3)
# Approximate linewidth: FWHM from Lorentzian fit
delta_lam = FSR_sweep * (1 - a_rt * t_c) / (np.pi * np.sqrt(a_rt * t_c))
Q_factor  = lam0 / delta_lam
finesse   = FSR_sweep / delta_lam

ax2 = axes[1].twinx()
axes[1].loglog(R_sweep, Q_factor, color='#e41a1c', lw=2, label='Q factor')
ax2.loglog(R_sweep, finesse, '--', color='#4daf4a', lw=2, label='Finesse')
axes[1].set_xlabel('Ring radius (μm)')
axes[1].set_ylabel('Q factor', color='#e41a1c')
ax2.set_ylabel('Finesse', color='#4daf4a')
axes[1].set_title('Q and finesse vs ring radius')
axes[1].grid(True, alpha=0.3)
lines1, lab1 = axes[1].get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, lab1+lab2)

plt.tight_layout(); plt.savefig('ip_ring.png', bbox_inches='tight'); plt.show()
print(f'FSR = {FSR_nm:.2f} nm  for R={R_um} μm, ng={ng}')
print(f'Q ≈ {lam0 / (FSR_nm * (1-a_rt*t_c)/(np.pi*np.sqrt(a_rt*t_c))):.0f}')

## §7 · Group velocity dispersion in Si nanowires

GVD $\beta_2$ is engineered by waveguide geometry — anomalous dispersion (negative $\beta_2$) enables soliton propagation and broadband GS phase retrieval.

In [ ]:
# GVD vs wavelength: qualitative model for Si strip waveguide
# β₂ = d²β/dω² = -(λ²/2πc) D   [ps²/m]
# We use material dispersion of Si + empirical waveguide correction

lam_um2  = np.linspace(1.2, 2.0, 400)
lam_nm2  = lam_um2 * 1e3
n_si2    = n_si(lam_um2)

dlam2    = lam_um2[1] - lam_um2[0]
D_mat2   = material_dispersion(n_si2, lam_um2)

# Waveguide dispersion: shifts ZDW toward shorter wavelength for narrow wires
# Empirical: D_wg ≈ -A/d² where d is height, A is a fit constant
heights  = [0.22, 0.30, 0.45]  # μm
A_wg     = 1200  # ps/(nm·km·μm²) — fit to numerical mode solver results

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lam_nm2, D_mat2, 'k--', lw=1.5, label='Material only', alpha=0.6)

for h, col in zip(heights, ['#377eb8','#e41a1c','#4daf4a']):
    D_wg   = -A_wg / h**2 * np.ones_like(lam_um2)  # rough approximation
    D_tot  = D_mat2 + D_wg
    ax.plot(lam_nm2, D_tot, lw=2, color=col, label=f'h={h*1000:.0f} nm wire')
    # ZDW
    try:
        zdw_i = np.where(np.diff(np.sign(D_tot)))[0][0]
        ax.axvline(lam_nm2[zdw_i], ls=':', color=col, alpha=0.7)
        ax.text(lam_nm2[zdw_i]+5, -2000, f'ZDW\n{lam_nm2[zdw_i]:.0f}nm',
                color=col, fontsize=8)
    except IndexError:
        pass

ax.axhline(0, color='k', lw=0.8)
ax.fill_between(lam_nm2, -5000, 0, alpha=0.04, color='blue', label='Normal (β₂>0)')
ax.fill_between(lam_nm2, 0, 5000, alpha=0.04, color='red',  label='Anomalous (β₂<0)')
ax.set_xlim(1200, 2000); ax.set_ylim(-5000, 3000)
ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('D [ps/(nm·km)]')
ax.set_title('Total GVD in Si nanowire waveguides (width=500 nm)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('ip_gvd_si.png', bbox_inches='tight'); plt.show()

## §8 · Mach-Zehnder modulator (electro-optic)

An MZM splits light into two arms, applies a voltage-controlled phase shift via the Pockels effect, then recombines:

$$E_{out} = \frac{E_{in}}{2}\left(e^{j\phi_1} + e^{j\phi_2}\right), \quad P_{out} = P_{in}\cos^2\!\left(\frac{\Delta\phi}{2}\right)$$

$V_\pi$ = voltage for $\pi$ phase shift (full on→off).  LiNbO₃: $V_\pi \approx 3$–5 V.  Si: $V_\pi \approx 10$–20 V.

In [ ]:
V_pi   = 5.0   # volts (LiNbO₃)
V_bias = V_pi / 2  # quadrature bias

V_rf   = np.linspace(-2 * V_pi, 2 * V_pi, 800)
dphi   = np.pi * (V_rf + V_bias) / V_pi
P_out  = np.cos(dphi / 2)**2

# QPSK drive: two MZMs in IQ configuration
t_sym  = np.linspace(0, 8, 800)  # normalized time
bits_I = np.sign(np.sin(2 * np.pi * t_sym))
bits_Q = np.sign(np.sin(2 * np.pi * t_sym + np.pi/4))
E_I    = np.cos(np.pi * bits_I / 2)
E_Q    = 1j * np.cos(np.pi * bits_Q / 2)
E_iq   = (E_I + E_Q) / np.sqrt(2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(V_rf, P_out, lw=2, color='#377eb8')
axes[0].axvline(0, ls='--', color='gray', alpha=0.6)
axes[0].axhline(0.5, ls=':', color='gray', alpha=0.6, label='Quadrature (0.5)')
axes[0].set_xlabel('Applied voltage V (V)')
axes[0].set_ylabel('Normalized output power')
axes[0].set_title(f'MZM transfer function  Vπ={V_pi} V')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_sym, np.real(E_iq), lw=1.5, color='#e41a1c', label='I')
axes[1].plot(t_sym, np.imag(E_iq), lw=1.5, color='#377eb8', label='Q')
axes[1].set_xlabel('Time (symbols)'); axes[1].set_ylabel('Field amplitude')
axes[1].set_title('IQ-MZM QPSK drive waveforms'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].scatter(np.real(E_iq), np.imag(E_iq), s=6, alpha=0.5, color='crimson')
axes[2].set_aspect('equal')
axes[2].set_xlabel('I'); axes[2].set_ylabel('Q')
axes[2].set_title('QPSK constellation from IQ-MZM')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(-1.5, 1.5); axes[2].set_ylim(-1.5, 1.5)

plt.tight_layout(); plt.savefig('ip_mzm.png', bbox_inches='tight'); plt.show()
print(f'Vπ = {V_pi} V  →  Vπ·L ≈ {V_pi*3:.1f} V·cm (LiNbO₃ traveling-wave)')

## Summary — First Terms of Integrated Photonics

| Component | Key parameter | Typical value (Si₃N₄) |
|-----------|--------------|----------------------|
| Slab waveguide | V number | V < π/2 for SM |
| Confinement | Γ | 0.7–0.9 |
| Dispersion | ZDW | ~1300 nm (Si), ~1700 nm (Si₃N₄) |
| Directional coupler | κL | π/4 for 3 dB |
| 90° hybrid | IQ outputs | Enables coherent receiver |
| Ring resonator | Q factor | 10⁵–10⁸ (Si₃N₄) |
| MZM | Vπ | 3–5 V (LiNbO₃), 10–20 V (Si) |

**Connection to GS phase retrieval:** the dispersive Fourier transform arm is a long dispersive waveguide or DCF fiber providing $|D| \gg 1$ — same dispersion physics as §3 and §7.  The 90° hybrid (§5) is the alternative (coherent) baseline the GS system is trying to replace.